# COCO Evaluation 101

This notebook walks through COCO object detection evaluation with **hotcoco** — a drop-in replacement for pycocotools that runs 11-26x faster.

**Getting started** — run a standard eval, interpret the 12 metrics, see per-class AP

**Understand your model** — TIDE errors, confusion matrices, calibration, per-image diagnostics

**Compare & slice** — side-by-side model comparison, metrics by image subset

**Dataset tools** — stats, filter, split, sample, interactive browse

**Integration** — drop-in pycocotools replacement, experiment tracker logging

---

## Install

In [ ]:
%pip install hotcoco[plot]

## 1. Quickstart — synthetic data

No downloads required. We'll build a tiny dataset in memory to verify everything works.

In [ ]:
from hotcoco import COCO, COCOeval

# Minimal in-memory COCO dataset
dataset = {
    "images": [{"id": 1, "width": 640, "height": 480}],
    "categories": [{"id": 1, "name": "cat"}, {"id": 2, "name": "dog"}],
    "annotations": [
        {"id": 1, "image_id": 1, "category_id": 1, "bbox": [10, 20, 100, 80],  "area": 8000,  "iscrowd": 0},
        {"id": 2, "image_id": 1, "category_id": 2, "bbox": [200, 150, 120, 90], "area": 10800, "iscrowd": 0},
    ],
}

detections = [
    {"image_id": 1, "category_id": 1, "bbox": [12, 22, 98, 78],  "score": 0.95},
    {"image_id": 1, "category_id": 2, "bbox": [205, 155, 115, 85], "score": 0.80},
    {"image_id": 1, "category_id": 1, "bbox": [300, 300, 50, 50], "score": 0.30},  # FP
]

coco_gt = COCO(dataset)
coco_dt = coco_gt.load_res(detections)

ev = COCOeval(coco_gt, coco_dt, "bbox")
ev.evaluate()
ev.accumulate()
ev.summarize()

## 2. Full COCO val2017 evaluation

Download COCO val2017 annotations and a detections file, then swap in the paths below.

```bash
# Download annotations (if you don't have them)
wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip
unzip annotations_trainval2017.zip
```

Any model that outputs COCO-format JSON detections works here.

In [ ]:
GT_PATH = "annotations/instances_val2017.json"
DT_PATH = "detections.json"  # your model's output

coco_gt = COCO(GT_PATH)
coco_dt = coco_gt.load_res(DT_PATH)

ev = COCOeval(coco_gt, coco_dt, "bbox")
ev.run()  # shorthand for evaluate() + accumulate() + summarize()

### Understanding the 12 metrics

| Metric | Description |
|--------|-------------|
| **AP** | mAP averaged over IoU thresholds 0.50:0.05:0.95 |
| **AP50** | mAP at IoU ≥ 0.50 (PASCAL VOC style) |
| **AP75** | mAP at IoU ≥ 0.75 (strict) |
| **APs / APm / APl** | AP for small / medium / large objects |
| **AR1 / AR10 / AR100** | Max recall given 1, 10, 100 detections per image |
| **ARs / ARm / ARl** | AR for small / medium / large objects |

Small = area < 32², medium = 32²–96², large = area > 96².

In [ ]:
# Get results as a dict — useful for logging or further analysis
results = ev.get_results()
print(results)
# {'AP': 0.382, 'AP50': 0.584, 'AP75': 0.412, ...}

## 3. Per-class AP

`per_class=True` adds one entry per category, keyed as `AP/{name}`.

In [ ]:
results = ev.get_results(per_class=True)

# Pull out per-class AP and sort descending
per_class = {k: v for k, v in results.items() if k.startswith("AP/")}
for cat, ap in sorted(per_class.items(), key=lambda x: -x[1]):
    print(f"  {cat:<30} {ap:.3f}")

---

# Understand your model

The sections below use a richer synthetic dataset (4 images, 3 categories) with
intentional errors — misclassifications, false positives, missed objects — so each
diagnostic tool has something to find.

In [ ]:
from hotcoco import COCO, COCOeval
import hotcoco.plot as plot

dataset = {
    "images": [
        {"id": 1, "width": 640, "height": 480},
        {"id": 2, "width": 640, "height": 480},
        {"id": 3, "width": 640, "height": 480},
        {"id": 4, "width": 640, "height": 480},
    ],
    "categories": [
        {"id": 1, "name": "cat"},
        {"id": 2, "name": "dog"},
        {"id": 3, "name": "bird"},
    ],
    "annotations": [
        {"id": 1, "image_id": 1, "category_id": 1, "bbox": [10, 20, 100, 80],   "area": 8000,  "iscrowd": 0},
        {"id": 2, "image_id": 1, "category_id": 2, "bbox": [200, 150, 120, 90],  "area": 10800, "iscrowd": 0},
        {"id": 3, "image_id": 2, "category_id": 1, "bbox": [50, 50, 80, 60],     "area": 4800,  "iscrowd": 0},
        {"id": 4, "image_id": 2, "category_id": 3, "bbox": [300, 200, 60, 40],   "area": 2400,  "iscrowd": 0},
        {"id": 5, "image_id": 3, "category_id": 2, "bbox": [100, 100, 150, 120], "area": 18000, "iscrowd": 0},
        {"id": 6, "image_id": 3, "category_id": 3, "bbox": [400, 50, 70, 50],    "area": 3500,  "iscrowd": 0},
        {"id": 7, "image_id": 4, "category_id": 1, "bbox": [20, 30, 90, 70],     "area": 6300,  "iscrowd": 0},
        {"id": 8, "image_id": 4, "category_id": 2, "bbox": [250, 180, 110, 85],  "area": 9350,  "iscrowd": 0},
    ],
}

detections = [
    # Image 1: good cat, good dog, FP bird
    {"image_id": 1, "category_id": 1, "bbox": [12, 22, 98, 78],    "score": 0.95},
    {"image_id": 1, "category_id": 2, "bbox": [205, 155, 115, 85], "score": 0.88},
    {"image_id": 1, "category_id": 3, "bbox": [400, 400, 30, 30],  "score": 0.40},  # FP
    # Image 2: cat classified as dog (Cls error), good bird
    {"image_id": 2, "category_id": 2, "bbox": [52, 48, 78, 62],    "score": 0.85},  # wrong class
    {"image_id": 2, "category_id": 3, "bbox": [302, 198, 58, 42],  "score": 0.72},
    # Image 3: good dog, missed bird, BG false positive
    {"image_id": 3, "category_id": 2, "bbox": [105, 105, 145, 115], "score": 0.91},
    {"image_id": 3, "category_id": 1, "bbox": [500, 400, 40, 40],  "score": 0.35},  # FP
    # Image 4: good cat, dog slightly off, duplicate cat
    {"image_id": 4, "category_id": 1, "bbox": [22, 32, 88, 68],    "score": 0.93},
    {"image_id": 4, "category_id": 2, "bbox": [280, 210, 100, 75], "score": 0.60},
    {"image_id": 4, "category_id": 1, "bbox": [25, 35, 85, 65],    "score": 0.50},  # duplicate
]

coco_gt = COCO(dataset)
coco_dt = coco_gt.load_res(detections)

ev = COCOeval(coco_gt, coco_dt, "bbox")
ev.evaluate()
ev.accumulate()
ev.summarize()

## 4. TIDE error analysis

TIDE decomposes every false positive and false negative into six error types,
then measures each one's impact on AP:

| Error | Meaning |
|-------|---------|
| **Loc** | Correct class, box IoU too low |
| **Cls** | Correct box, wrong class |
| **Dupe** | Duplicate detection of an already-matched GT |
| **Bkg** | Detection with no nearby GT (pure background) |
| **Both** | Wrong class *and* wrong box |
| **Miss** | GT with no detection attempting to match it |

In [ ]:
tide = ev.tide_errors()

print(f"Base AP: {tide['ap_base']:.4f}\n")
error_types = ["Loc", "Cls", "Dupe", "Bkg", "Both", "Miss"]
for name in sorted(error_types, key=lambda k: -tide["delta_ap"].get(k, 0)):
    delta = tide["delta_ap"].get(name, 0)
    count = tide["counts"].get(name, 0)
    print(f"  {name:<8} ΔAP={delta:.4f}  n={count}")

In [ ]:
fig, ax = plot.tide_errors(tide)

The largest ΔAP tells you where to focus:
- High **Loc** → improve bounding box regression
- High **Cls** → improve classification head or NMS
- High **Miss** → model misses objects entirely; try lower confidence threshold
- High **Bkg** → too many false positives; tighten confidence threshold

## 5. Confusion matrix

Which categories does your model confuse? The matrix is (K+1)×(K+1) — the extra
row/column captures background (unmatched detections and missed GTs).

In [ ]:
cm = ev.confusion_matrix(iou_thr=0.5)

print(f"Categories: {cm['cat_names']}")
print(f"Matrix shape: {cm['matrix'].shape}  (last row/col = background)\n")
print(cm['normalized'].round(2))

In [ ]:
fig, ax = plot.confusion_matrix(cm)

## 6. Confidence calibration

Is your model's confidence score meaningful? A well-calibrated detector should be
correct 90% of the time when it reports 0.9 confidence. ECE (Expected Calibration
Error) measures the average gap; MCE is the worst single bin.

In [ ]:
cal = ev.calibration(n_bins=5, iou_threshold=0.5)

print(f"ECE: {cal['ece']:.4f}")
print(f"MCE: {cal['mce']:.4f}\n")
for b in cal['bins']:
    if b['count'] > 0:
        print(f"  [{b['bin_lower']:.1f}, {b['bin_upper']:.1f})  "
              f"conf={b['avg_confidence']:.2f}  acc={b['avg_accuracy']:.2f}  n={b['count']}")

In [ ]:
fig, ax = plot.reliability_diagram(ev, n_bins=5)

## 7. Per-image diagnostics & label errors

Go beyond aggregate metrics — see which images are hardest and find annotation mistakes.

Each image gets an **error profile** (`perfect`, `fp_heavy`, `fn_heavy`, or `mixed`).
Label error detection flags suspected wrong labels and missing annotations.

In [ ]:
diag = ev.image_diagnostics(iou_thr=0.5, score_thr=0.3)

for img_id, s in sorted(diag['img_summary'].items()):
    print(f"  Image {img_id}: F1={s['f1']:.2f}  "
          f"TP={s['tp']} FP={s['fp']} FN={s['fn']}  [{s['error_profile']}]")

In [ ]:
# Suspected label errors — useful for data curation at scale
for err in diag['label_errors']:
    print(f"  Image {err['image_id']}: {err['type']} — "
          f"dt={err['dt_category']} (score={err['dt_score']:.2f})")

## 8. F-scores

`f_scores()` finds the confidence threshold that maximises F-beta for each
(IoU threshold, category), then averages. Useful when you care about the
precision/recall balance at deployment, not just the area under the curve.

In [ ]:
ev.f_scores()          # F1 (balanced)
ev.f_scores(beta=0.5)  # favour precision
ev.f_scores(beta=2.0)  # favour recall

## 9. PR curves & per-category AP

`hotcoco.plot` provides publication-quality charts with three built-in themes:
`"warm-slate"` (default), `"scientific-blue"`, and `"ember"`.

All plot functions return `(fig, ax)` and accept `save_path=` for direct export.

In [ ]:
fig, ax = plot.pr_curve_iou_sweep(ev)

In [ ]:
results = ev.results(per_class=True)
fig, ax = plot.per_category_ap(results)

In [ ]:
# Switch themes or enable paper_mode for LaTeX/slides
fig, ax = plot.pr_curve_iou_sweep(ev, theme="scientific-blue")
# fig, ax = plot.pr_curve_iou_sweep(ev, paper_mode=True, save_path="pr_curve.pdf")

---

# Compare & slice

## 10. Model comparison

Compare two models side-by-side with per-metric deltas and optional bootstrap
confidence intervals for statistical significance.

In [ ]:
from hotcoco import compare

# "Model B" — tighter boxes, higher scores (simulating a better model)
detections_b = [
    {"image_id": 1, "category_id": 1, "bbox": [11, 21, 99, 79],    "score": 0.97},
    {"image_id": 1, "category_id": 2, "bbox": [202, 152, 118, 88], "score": 0.92},
    {"image_id": 2, "category_id": 1, "bbox": [51, 49, 79, 61],    "score": 0.90},
    {"image_id": 2, "category_id": 3, "bbox": [301, 199, 59, 41],  "score": 0.80},
    {"image_id": 3, "category_id": 2, "bbox": [102, 102, 148, 118], "score": 0.94},
    {"image_id": 3, "category_id": 3, "bbox": [402, 52, 68, 48],   "score": 0.75},
    {"image_id": 4, "category_id": 1, "bbox": [21, 31, 89, 69],    "score": 0.96},
    {"image_id": 4, "category_id": 2, "bbox": [252, 182, 108, 83], "score": 0.85},
]

coco_dt_b = coco_gt.load_res(detections_b)
ev_b = COCOeval(coco_gt, coco_dt_b, "bbox")
ev_b.evaluate()
ev_b.accumulate()
ev_b.summarize()

cmp = compare(ev, ev_b, n_bootstrap=200)

print("Metric       Model A   Model B   Delta")
print("-" * 42)
for key in cmp['metric_keys'][:6]:
    a = cmp['metrics_a'][key]
    b = cmp['metrics_b'][key]
    d = cmp['deltas'][key]
    print(f"{key:<12} {a:>7.3f}   {b:>7.3f}   {d:>+.3f}")

In [ ]:
fig, ax = plot.comparison_bar(cmp)

## 11. Sliced evaluation

Break down metrics by image subsets — indoor vs outdoor, day vs night,
camera A vs B — without recomputing IoU matching.

In [ ]:
slices = {
    "easy":  [1, 4],  # images with strong detections
    "hard":  [2, 3],  # images with classification errors / misses
}

result = ev.slice_by(slices)

for name, data in result.items():
    ap = data.get("AP", 0)
    n = data["num_images"]
    print(f"  {name:<12} AP={ap:.3f}  ({n} images)")

---

# Dataset tools

## 12. Dataset inspection

`COCO` is also a full dataset API — query images, annotations, and categories.

In [ ]:
coco = COCO("annotations/instances_val2017.json")

cats = coco.load_cats(coco.get_cat_ids())
print([c["name"] for c in cats])

ann_ids = coco.get_ann_ids(img_ids=[139], cat_ids=[1])
anns = coco.load_anns(ann_ids)
print(anns)

## 13. Dataset operations

`COCO` objects support filter, split, sample, merge, and stats — all returning
new `COCO` instances. The original is never mutated.

In [ ]:
# Dataset summary
print(coco_gt.stats())

In [ ]:
# Filter to a single category
cats_only = coco_gt.filter(cat_ids=[1])
print(f"Filtered: {len(cats_only.get_img_ids())} images, "
      f"{len(cats_only.get_ann_ids())} annotations (cat only)")

# Train/val split (deterministic)
train, val = coco_gt.split(val_frac=0.5, seed=42)
print(f"Train: {len(train.get_img_ids())} images")
print(f"Val:   {len(val.get_img_ids())} images")

# Random sample
subset = coco_gt.sample(n=2, seed=42)
print(f"Sample: {len(subset.get_img_ids())} images")

## 14. Interactive browse

Launch a dataset browser directly in Jupyter. Requires images on disk
and the browse extras (`pip install hotcoco[browse]`).

The browser supports bbox/segm/keypoint overlays, GT and detection toggle,
and eval-aware mode with TP (green) / FP (red) / FN (yellow) coloring.

In [ ]:
# Uncomment to launch (requires images on disk):

# coco_gt.browse(image_dir="path/to/images/", dt=coco_dt)

# With eval overlay — TP/FP/FN coloring:
# coco_gt.browse(image_dir="path/to/images/", eval=ev)

# With sliced browsing:
# coco_gt.browse(image_dir="path/to/images/", eval=ev, slices=slices)

---

# Integration

## 15. Drop-in replacement for pycocotools

If you use Detectron2, Ultralytics, MMDetection, or any pipeline that imports
`pycocotools`, one call at startup replaces everything — no other changes needed.

In [ ]:
from hotcoco import init_as_pycocotools
init_as_pycocotools()

# These now resolve to hotcoco under the hood
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask

## 16. Segmentation and keypoint evaluation

Same API — just swap `"bbox"` for `"segm"` or `"keypoints"`.

In [ ]:
# Segmentation
coco_gt = COCO("annotations/instances_val2017.json")
coco_dt = coco_gt.load_res("segm_results.json")
ev = COCOeval(coco_gt, coco_dt, "segm")
ev.run()

# Keypoints
coco_gt = COCO("annotations/person_keypoints_val2017.json")
coco_dt = coco_gt.load_res("kpt_results.json")
ev = COCOeval(coco_gt, coco_dt, "keypoints")
ev.run()

## 17. Logging to experiment trackers

`prefix` prepends a path to all metric keys so they group cleanly in your tracker's UI.

In [ ]:
# Weights & Biases
import wandb
wandb.log(ev.get_results(prefix="val/bbox", per_class=True), step=epoch)

# MLflow
import mlflow
mlflow.log_metrics(ev.get_results(prefix="val/bbox"), step=epoch)

---

## Next steps

- [Full API reference](https://derekallman.github.io/hotcoco/api/cocoeval/)
- [LVIS federated evaluation](https://derekallman.github.io/hotcoco/user-guide/evaluation/#lvis)
- [CLI reference](https://derekallman.github.io/hotcoco/cli/) — `coco eval`, `coco stats`, `coco compare` from the terminal
- [GitHub](https://github.com/derekallman/hotcoco)